# Task 5 — Regional Node Network

Select regional hubs from the CoStar candidate pool using a capacity-aware set-cover MIP, then define the regional hub network topology using geographic and freight-interaction criteria.

**Notebook scope:** Tasks 5.1 through 5.6 (H/Z construction → MIP → analysis → network links → figures).

---

## Task 5.1 — Candidate Set H and Z Construction

Build the filtered hub candidate set **H** and the feasibility set **Z**.

### Steps
1. **H construction** — apply adaptive size floor; sparse-region fallback from broader pool
2. **Road accessibility β gate** — compute $d_h^{road}$ for each candidate; exclude the 10% most remote
3. **Z construction** — all $(h, r)$ pairs where Euclidean distance ≤ 150 miles (241 402 m)

### Key parameters
| Symbol | Value | Meaning |
| ------ | ----- | ------- |
| base floor | 100 000 sqft | Minimum `usable_available_space_sf` from primary pool |
| fallback floor | 50 000 sqft | Minimum sqft from supplementary pool for sparse regions |
| $\beta$ | 90th pct | Maximum $d_h^{road}$ allowed in H |
| 150 mi | 241 402 m | Maximum hub-to-region centroid distance for Z |
| 75 mi | 120 701 m | Z-neighbor radius for sparse-region check |

In [1]:
# ── Imports and paths ────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from shapely import STRtree
from pyproj import Transformer
from pathlib import Path

# Locate project root (contains both Data/ and Doc/), regardless of CWD
ROOT = Path(".").resolve()
for _ in range(4):
    if (ROOT / "Data").exists() and (ROOT / "Doc").exists():
        break
    ROOT = ROOT.parent

DATA_T3    = ROOT / "Data/Task3"
DATA_T4    = ROOT / "Data/Task4/processed"
DATA_T5    = ROOT / "Data/Task5"
CACHE      = DATA_T5 / "cache"
CACHE.mkdir(parents=True, exist_ok=True)

# Constants
BASE_FLOOR     = 100_000          # sqft — base filter from primary pool
FALLBACK_FLOOR =  50_000          # sqft — fallback for sparse regions
SPARSE_N       = 2                # min in-region candidates before fallback check
SPARSE_RADIUS  = 75 * 1609.34    # 75 miles in metres
MAX_DIST_M     = 150 * 1609.34   # 150 miles in metres  (= 241 402 m)
BETA_PCTILE    = 90               # road accessibility gate percentile
EXEMPT_REGIONS = {43, 19}        # confirmed external Z coverage; no fallback needed

# CRS transformer: WGS-84 → EPSG:9311 (equal-area, metres)
PROJ = Transformer.from_crs("EPSG:4326", "EPSG:9311", always_xy=True)

print("Imports OK")
print(f"ROOT: {ROOT}")

Imports OK
ROOT: /home/hty/Documents/Working Projects/ISYE6339_Case2


---

## 5.1.1 — H Construction: Adaptive Size Floor

**Base filter** — retain candidates from `primary_regional_hub_candidates.csv` with:
$$s_h \geq 100{,}000 \text{ sqft}$$

Because Task 4 already enforced $s_h \geq 200{,}000$ sqft on the primary pool, this filter effectively keeps every row and merely documents the floor used in Task 5.

**Sparse-region fallback** — for region $r$ with fewer than 2 in-region candidates *and* fewer than 2 H-candidates within 75 miles of its centroid, pull additional candidates from the broader `preprocessed_capacity_location.csv` pool where $s_h \geq 50{,}000$ sqft.

Regions 43 (ME) and 19 (rural PA/WV) are exempt: both have ≥ 160 H-candidates within 150 miles via neighboring regions, providing sufficient Z coverage without fallback.

In [2]:
# ── Load primary candidate pool ───────────────────────────────────────────────
primary = pd.read_csv(DATA_T4 / "primary_regional_hub_candidates.csv")
print(f"Primary pool  : {len(primary):>6,} rows")
print(f"Columns       : {list(primary.columns)}")
print(f"sqft range    : {primary['usable_available_space_sf'].min():,.0f} – "
      f"{primary['usable_available_space_sf'].max():,.0f}")

Primary pool  :  1,862 rows
Columns       : ['candidate_id', 'source_state', 'facility_name', 'city', 'county_fips', 'county_name', 'region_id', 'latitude', 'longitude', 'secondary_type', 'building_class', 'building_status', 'year_built', 'usable_available_space_sf', 'number_loading_docks', 'availability_class', 'is_directly_usable_by_status', 'meets_min_rba_200k', 'is_primary_regional_hub_candidate']
sqft range    : 200,000 – 500,000


In [3]:
# ── Apply base filter ─────────────────────────────────────────────────────────
H = primary[primary['usable_available_space_sf'] >= BASE_FLOOR].copy().reset_index(drop=True)
print(f"After base filter (≥{BASE_FLOOR:,} sqft): {len(H):,} candidates")

# Project to EPSG:9311
H_x, H_y = PROJ.transform(H['longitude'].values, H['latitude'].values)
H['x_m'] = H_x
H['y_m'] = H_y

# Per-region candidate count (before fallback)
region_counts_pre = H.groupby('region_id').size().rename('n_base')
print(f"\nRegion candidate count (base):")
print(region_counts_pre.describe().round(1))

After base filter (≥100,000 sqft): 1,862 candidates

Region candidate count (base):
count    50.0
mean     37.2
std      24.9
min       4.0
25%      15.0
50%      30.0
75%      60.0
max      98.0
Name: n_base, dtype: float64


In [4]:
# ── Sparse-region fallback ────────────────────────────────────────────────────
region_metrics = pd.read_csv(DATA_T3 / "outputs/region_metrics.csv")
region_metrics = region_metrics.sort_values('region_id').reset_index(drop=True)

H_coords = H[['x_m', 'y_m']].values

sparse_needs_fallback = []
for _, row in region_metrics.iterrows():
    r_id = int(row['region_id'])
    if r_id in EXEMPT_REGIONS:
        continue
    in_region = (H['region_id'] == r_id).sum()
    if in_region < SPARSE_N:
        # Count any H candidate within 75 miles of this region centroid
        dists = np.sqrt(
            (H_coords[:, 0] - row['centroid_x_m'])**2 +
            (H_coords[:, 1] - row['centroid_y_m'])**2
        )
        nearby = int((dists <= SPARSE_RADIUS).sum())
        if nearby < SPARSE_N:
            sparse_needs_fallback.append(r_id)
            print(f"  Region {r_id:2d}: {in_region} in-region, {nearby} within 75 mi → FALLBACK")

if not sparse_needs_fallback:
    print("No regions need fallback candidates.")
else:
    print(f"\nRegions needing fallback: {sparse_needs_fallback}")

No regions need fallback candidates.


In [5]:
# ── Pull fallback candidates (if any) ────────────────────────────────────────
if sparse_needs_fallback:
    preprocessed = pd.read_csv(DATA_T4 / "preprocessed_capacity_location.csv")
    fallback = preprocessed[
        preprocessed['region_id'].isin(sparse_needs_fallback) &
        (preprocessed['usable_available_space_sf'] >= FALLBACK_FLOOR) &
        (~preprocessed['candidate_id'].isin(H['candidate_id']))
    ].copy()
    if len(fallback) > 0:
        fb_x, fb_y = PROJ.transform(fallback['longitude'].values, fallback['latitude'].values)
        fallback['x_m'] = fb_x
        fallback['y_m'] = fb_y
        H = pd.concat([H, fallback], ignore_index=True)
        print(f"Added {len(fallback)} fallback candidates for regions {sparse_needs_fallback}")
    else:
        print("Fallback pool had no matching candidates.")

H = H.reset_index(drop=True)
H['h_idx'] = H.index          # integer index used in Z pairs

print(f"\n|H| after base filter + fallback = {len(H):,}")
print(f"sqft range: {H['usable_available_space_sf'].min():,.0f} – "
      f"{H['usable_available_space_sf'].max():,.0f}")


|H| after base filter + fallback = 1,862
sqft range: 200,000 – 500,000


---

## 5.1.2 — Road Accessibility Pre-filter (β Gate)

For each candidate $h \in H$, compute the minimum Euclidean distance in EPSG:9311 to any US interstate segment:

$$d_h^{road} = \min_{s \in \mathcal{S}_{int}} \|\mathbf{x}_h - \text{proj}(\mathbf{x}_h, s)\|_2$$

where $\mathcal{S}_{int}$ is the set of US interstate segments (COUNTRY = 2, CLASS = 1) from `North_American_Roads.shp`, projected to EPSG:9311.

The road accessibility threshold is set at:
$$\beta = \text{pct}_{90}(d_h^{road} \mid h \in H)$$

Candidates with $d_h^{road} > \beta$ are excluded from H as a hard pre-filter (Constraint 5 in the MIP), keeping the model size smaller than if β were implemented as a MIP variable.

In [6]:
# ── Load US interstates (or from cache) ───────────────────────────────────────
int_cache = CACHE / "us_interstates.parquet"

if int_cache.exists():
    interstates = gpd.read_parquet(int_cache)
    print(f"Loaded US interstates from cache: {len(interstates):,} segments")
else:
    ROADS_SHP = DATA_T3 / "raw/roads/North_American_Roads.shp"
    print(f"Reading {ROADS_SHP} ...")
    roads = gpd.read_file(ROADS_SHP)
    print(f"Roads columns: {list(roads.columns)}")
    print(f"COUNTRY dtype: {roads['COUNTRY'].dtype}, sample: {roads['COUNTRY'].unique()[:5]}")
    print(f"CLASS   dtype: {roads['CLASS'].dtype},   sample: {roads['CLASS'].unique()[:5]}")

    # Coerce to int if stored as float/string
    roads['COUNTRY'] = pd.to_numeric(roads['COUNTRY'], errors='coerce')
    roads['CLASS']   = pd.to_numeric(roads['CLASS'],   errors='coerce')

    mask = (roads['COUNTRY'] == 2) & (roads['CLASS'] == 1)
    interstates = roads[mask].to_crs("EPSG:9311")
    interstates.to_parquet(int_cache)
    print(f"Cached {len(interstates):,} US interstate segments → {int_cache}")

Reading /home/hty/Documents/Working Projects/ISYE6339_Case2/Data/Task3/raw/roads/North_American_Roads.shp ...


Roads columns: ['ID', 'DIR', 'LENGTH', 'LINKID', 'COUNTRY', 'JURISCODE', 'JURISNAME', 'ROADNUM', 'ROADNAME', 'ADMIN', 'SURFACE', 'LANES', 'SPEEDLIM', 'CLASS', 'NHS', 'BORDER', 'geometry']
COUNTRY dtype: int32, sample: [1 2 3]
CLASS   dtype: int64,   sample: [2 5 3 4 6]


Cached 100,009 US interstate segments → /home/hty/Documents/Working Projects/ISYE6339_Case2/Data/Task5/cache/us_interstates.parquet


In [7]:
# ── Compute d_h^road via STRtree nearest-neighbour ────────────────────────────
d_road_cache = CACHE / "d_road.npy"
H_idx_cache  = CACHE / "H_pre_beta.parquet"

if d_road_cache.exists() and H_idx_cache.exists():
    d_road = np.load(d_road_cache)
    H_pre  = pd.read_parquet(H_idx_cache)
    # Reload H from pre-beta snapshot so h_idx aligns
    H      = H_pre.copy()
    print(f"Loaded d_road from cache: {len(d_road):,} values")
else:
    int_geoms  = interstates.geometry.values
    strtree    = STRtree(int_geoms)

    H_points   = shapely.points(H['x_m'].values, H['y_m'].values)
    nearest_ix = strtree.nearest(H_points)
    d_road     = shapely.distance(H_points, int_geoms[nearest_ix])

    np.save(d_road_cache, d_road)
    H['d_road_m']     = d_road
    H['d_road_miles'] = d_road / 1609.34
    H.to_parquet(H_idx_cache, index=False)
    print(f"Computed and cached d_road for {len(d_road):,} candidates")

H['d_road_m']     = d_road
H['d_road_miles'] = d_road / 1609.34

print(f"\nd_road distribution (metres):")
print(pd.Series(d_road).describe().round(1))

Computed and cached d_road for 1,862 candidates

d_road distribution (metres):
count      1862.0
mean       5585.6
std       11573.3
min          44.0
25%         763.5
50%        1953.7
75%        4929.9
max      127582.7
dtype: float64


In [8]:
# ── Apply β gate ─────────────────────────────────────────────────────────────
beta_m     = float(np.percentile(d_road, BETA_PCTILE))
beta_miles = beta_m / 1609.34

n_before = len(H)
H = H[H['d_road_m'] <= beta_m].copy().reset_index(drop=True)
H['h_idx'] = H.index
n_excluded = n_before - len(H)

print(f"β = {BETA_PCTILE}th percentile d_road = {beta_m:,.0f} m  ({beta_miles:.2f} miles)")
print(f"Excluded  : {n_excluded:,} candidates (d_road > β)")
print(f"|H| final : {len(H):,} candidates")

# Per-region breakdown after β
region_counts_post = H.groupby('region_id').size().rename('n_post_beta')
print(f"\nRegion candidate count (post β-gate):")
print(region_counts_post.describe().round(1))

β = 90th percentile d_road = 12,831 m  (7.97 miles)
Excluded  : 187 candidates (d_road > β)
|H| final : 1,675 candidates

Region candidate count (post β-gate):
count    49.0
mean     34.2
std      24.7
min       4.0
25%      13.0
50%      26.0
75%      52.0
max      92.0
Name: n_post_beta, dtype: float64


---

## 5.1.3 — Z Construction

The feasibility set $Z$ contains all hub–region pairs $(h, r)$ where the Euclidean distance from hub $h$ to region $r$'s centroid (both in EPSG:9311) is at most 150 miles:

$$Z = \left\{ (h, r) \in H \times R \;\middle|\; \|\mathbf{x}_h - \boldsymbol{\mu}_r\|_2 \leq 241{,}402 \text{ m} \right\}$$

where $\boldsymbol{\mu}_r = (\text{centroid\_x\_m}_r, \text{centroid\_y\_m}_r)$ from `region_metrics.csv`.

The pairwise distance matrix $D \in \mathbb{R}^{|H| \times 50}$ is computed once via NumPy broadcasting and cached for reuse in Task 5.2.

In [9]:
# ── Build Z (h, r) feasibility pairs ─────────────────────────────────────────
Z_cache      = CACHE / "Z_pairs.npy"
dist_hr_path = CACHE / "dist_hr.npy"

# Region centroids ordered by region_id (0..49)
region_metrics = region_metrics.sort_values('region_id').reset_index(drop=True)
R_ids    = region_metrics['region_id'].values            # shape (50,)
R_coords = region_metrics[['centroid_x_m', 'centroid_y_m']].values  # (50, 2)

# Hub coordinates
H_coords = H[['x_m', 'y_m']].values                     # (|H|, 2)

# Pairwise Euclidean distance via broadcasting: (|H|, 50)
diffs   = H_coords[:, np.newaxis, :] - R_coords[np.newaxis, :, :]   # (|H|, 50, 2)
dist_hr = np.sqrt((diffs ** 2).sum(axis=2))                          # (|H|, 50)

# Z: indices where distance ≤ 150 miles
h_idx_z, r_idx_z = np.where(dist_hr <= MAX_DIST_M)
Z = np.column_stack([h_idx_z, r_idx_z]).astype(np.int32)   # (|Z|, 2)

np.save(dist_hr_path, dist_hr)
np.save(Z_cache, Z)

print(f"|H| = {len(H):,}")
print(f"|R| = {len(R_ids)}")
print(f"|Z| = {len(Z):,} feasible (h, r) pairs")
print(f"Mean Z-pairs per region : {len(Z)/len(R_ids):,.0f}")
print(f"Mean Z-pairs per hub    : {len(Z)/len(H):.1f}")

|H| = 1,675
|R| = 50
|Z| = 25,667 feasible (h, r) pairs
Mean Z-pairs per region : 513
Mean Z-pairs per hub    : 15.3


In [10]:
# ── Verify full region coverage in Z ─────────────────────────────────────────
covered_regions = set(r_idx_z.tolist())
all_region_ix   = set(range(len(R_ids)))
uncovered_ix    = all_region_ix - covered_regions

if uncovered_ix:
    uncovered_ids = [int(R_ids[i]) for i in uncovered_ix]
    print(f"WARNING: {len(uncovered_ix)} region(s) have NO hub in Z: region_ids = {uncovered_ids}")
else:
    print(f"✓ All {len(R_ids)} regions have ≥ 1 hub in Z")

# Per-region Z hub count
z_per_region = pd.Series(r_idx_z).value_counts().sort_index()
print(f"\nZ-hubs per region  min={z_per_region.min():,}  "
      f"median={z_per_region.median():,.0f}  max={z_per_region.max():,}")

# Special check for exempt regions 43 and 19
for rid in sorted(EXEMPT_REGIONS):
    r_pos = int(np.where(R_ids == rid)[0][0])
    z_count = int((r_idx_z == r_pos).sum())
    print(f"  Region {rid} (exempt) : {z_count} Z-hubs from external candidates")

✓ All 50 regions have ≥ 1 hub in Z

Z-hubs per region  min=13  median=475  max=944
  Region 19 (exempt) : 598 Z-hubs from external candidates
  Region 43 (exempt) : 13 Z-hubs from external candidates


---

## 5.1.4 — Save Outputs and Summary

In [11]:
# ── Persist final H dataframe ─────────────────────────────────────────────────
H_out_path = CACHE / "H_candidates.parquet"
H.to_parquet(H_out_path, index=False)

# Persist β scalar
np.save(CACHE / "beta_m.npy", np.array([beta_m]))

print("Outputs saved:")
print(f"  H_candidates.parquet  → {H_out_path}  ({len(H):,} rows)")
print(f"  Z_pairs.npy           → {CACHE / 'Z_pairs.npy'}  ({len(Z):,} pairs)")
print(f"  dist_hr.npy           → {CACHE / 'dist_hr.npy'}  shape {dist_hr.shape}")
print(f"  beta_m.npy            → {CACHE / 'beta_m.npy'}  β={beta_m:,.0f} m")

# Human-readable H summary
h_summary = H.groupby('region_id').agg(
    n_candidates=('h_idx', 'count'),
    min_sqft=('usable_available_space_sf', 'min'),
    max_sqft=('usable_available_space_sf', 'max'),
    median_d_road_miles=('d_road_miles', 'median')
).reset_index()
h_summary.to_csv(CACHE / "H_region_summary.csv", index=False)
print(f"  H_region_summary.csv  → {CACHE / 'H_region_summary.csv'}")

Outputs saved:
  H_candidates.parquet  → /home/hty/Documents/Working Projects/ISYE6339_Case2/Data/Task5/cache/H_candidates.parquet  (1,675 rows)
  Z_pairs.npy           → /home/hty/Documents/Working Projects/ISYE6339_Case2/Data/Task5/cache/Z_pairs.npy  (25,667 pairs)
  dist_hr.npy           → /home/hty/Documents/Working Projects/ISYE6339_Case2/Data/Task5/cache/dist_hr.npy  shape (1675, 50)
  beta_m.npy            → /home/hty/Documents/Working Projects/ISYE6339_Case2/Data/Task5/cache/beta_m.npy  β=12,831 m
  H_region_summary.csv  → /home/hty/Documents/Working Projects/ISYE6339_Case2/Data/Task5/cache/H_region_summary.csv


---

## 5.1.5 — Sanity Checks

Expected bounds drawn from `Doc/Task.md` and project parameters:

| Check | Bound | Source |
| ----- | ----- | ------ |
| `|H|` size | 500 – 2 000 | After base filter + β removal |
| All regions covered in Z | 50/50 | Constraint 1 feasibility |
| β > 0 | always | 90th percentile of positive distances |
| Z index range | h < |H|, r < 50 | Index integrity |
| d_road ≥ 0 | always | Physical distance |
| sqft ≥ BASE_FLOOR in H | always | H construction guarantee |
| max(dist_hr) for covered pairs ≤ MAX_DIST_M | ≤ 241 402 m | Z construction rule |

In [12]:
# ── Sanity checks ─────────────────────────────────────────────────────────────
errors = []

# 1. H size
if not (500 <= len(H) <= 2000):
    errors.append(f"|H| = {len(H)} outside expected range [500, 2000]")
else:
    print(f"✓ |H| = {len(H):,}  (expected 500–2 000)")

# 2. Full region coverage
if uncovered_ix:
    errors.append(f"Uncovered regions in Z: {[int(R_ids[i]) for i in uncovered_ix]}")
else:
    print(f"✓ All 50 regions covered in Z")

# 3. β positive
if beta_m <= 0:
    errors.append(f"β = {beta_m:.1f} m ≤ 0")
else:
    print(f"✓ β = {beta_m:,.0f} m ({beta_miles:.2f} mi) > 0")

# 4. Z index integrity
if len(Z) > 0:
    if Z[:, 0].max() >= len(H):
        errors.append("Z contains h_idx >= |H|")
    if Z[:, 1].max() >= len(R_ids):
        errors.append("Z contains r_idx >= |R|")
    if not (Z[:, 0].min() >= 0 and Z[:, 1].min() >= 0):
        errors.append("Z contains negative index")
    if not errors:
        print(f"✓ Z index range: h ∈ [0,{Z[:,0].max()}], r ∈ [0,{Z[:,1].max()}]")

# 5. d_road ≥ 0
if (H['d_road_m'] < 0).any():
    errors.append("Negative d_road values in H")
else:
    print(f"✓ All d_road ≥ 0  "
          f"(range: {H['d_road_m'].min():.0f} – {H['d_road_m'].max():,.0f} m)")

# 6. Size floor
if (H['usable_available_space_sf'] < BASE_FLOOR).any():
    errors.append(f"H contains candidates below BASE_FLOOR={BASE_FLOOR:,}")
else:
    print(f"✓ All H candidates have sqft ≥ {BASE_FLOOR:,}  "
          f"(median: {H['usable_available_space_sf'].median():,.0f})")

# 7. Z pair distances all ≤ MAX_DIST_M
if len(Z) > 0:
    z_dists = dist_hr[Z[:, 0], Z[:, 1]]
    if z_dists.max() > MAX_DIST_M + 1:   # +1 m tolerance for float arithmetic
        errors.append(f"Z contains pair with dist {z_dists.max():.0f} m > {MAX_DIST_M:.0f} m")
    else:
        print(f"✓ All Z pair distances ≤ {MAX_DIST_M/1609.34:.0f} mi  "
              f"(max in Z: {z_dists.max()/1609.34:.1f} mi)")

# ── Final verdict ──
print()
if errors:
    for e in errors:
        print(f"  FAIL: {e}")
    raise AssertionError(f"{len(errors)} sanity check(s) failed — see above.")
else:
    print("══════════════════════════════════════")
    print("  All sanity checks PASSED — Task 5.1")
    print("══════════════════════════════════════")

✓ |H| = 1,675  (expected 500–2 000)
✓ All 50 regions covered in Z
✓ β = 12,831 m (7.97 mi) > 0
✓ Z index range: h ∈ [0,1674], r ∈ [0,49]
✓ All d_road ≥ 0  (range: 44 – 12,820 m)
✓ All H candidates have sqft ≥ 100,000  (median: 306,250)
✓ All Z pair distances ≤ 150 mi  (max in Z: 150.0 mi)

══════════════════════════════════════
  All sanity checks PASSED — Task 5.1
══════════════════════════════════════


---

## Milestone — Task 5.1 Complete

| Output | Path | Description |
| ------ | ---- | ----------- |
| `H_candidates.parquet` | `Data/Task5/cache/` | Final filtered hub candidate set with d_road |
| `Z_pairs.npy` | `Data/Task5/cache/` | (h_idx, r_idx) pairs — feasibility set Z |
| `dist_hr.npy` | `Data/Task5/cache/` | Full (|H| × 50) distance matrix for Task 5.2 |
| `beta_m.npy` | `Data/Task5/cache/` | Scalar β value |
| `H_region_summary.csv` | `Data/Task5/cache/` | Per-region candidate count and sqft stats |

**Next:** Task 5.2 — Objective Coefficients and Capacity Parameters (pre-compute ĉ_hr, cap_rhs, Q̄, T̄)